Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 3/4: Архитектура RAG → Урок 6/6: Сборка RAG-пайплайна с LangChain]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/b36d9f5f-3f26-4fbb-bd65-71d33c4523ef/lessons/827f1ac2-fe38-474d-843b-7f4ea53670d3/#8a453b5b-ef1e-402b-93fa-8951c603da30 )

In [1]:
from collections.abc import Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
from pypdf import PdfReader


class PypdfPageLoader(BaseLoader):
    """Читает PDF постранично и возвращает объекты Document."""

    def __init__(self, file_path: str):
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:
        """Читает страницы лениво, сохраняя источник и номер страницы."""
        reader = PdfReader(self.file_path)
        file_metadata = {
            key.removeprefix("/").lower(): value
            for key, value in (reader.metadata or {}).items()
        }

        for page_number, page in enumerate(reader.pages):
            yield Document(
                page_content=page.extract_text() or "",
                metadata={
                    **file_metadata,
                    "source": self.file_path,
                    "page": page_number,
                    "total_pages": len(reader.pages),
                },
            )

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Пропуск 1: загрузите velta_guide.pdf — получите список Document
loader = PypdfPageLoader("velta_guide.pdf")
docs = loader.load()

print("Страниц:", len(docs))
print("Метаданные:", docs[0].metadata["source"], "| стр.", docs[0].metadata["page"])

# Пропуск 2: нарежьте документы на чанки — размер 800, перекрытие 120
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(docs)
print("Чанков:", len(chunks))

# Эмбеддер — multilingual-e5-base.
# Сама обёртка не добавит префиксы e5 — пробрасываем их явно
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "prompt": "passage: "},
    query_encode_kwargs={"normalize_embeddings": True, "prompt": "query: "},
)

# Пропуск 3: соберите базу из чанков; метрика — косинус
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    collection_metadata={"hnsw:space": "cosine"}
)

# У обёртки нет публичного счётчика записей — заглядываем во внутреннюю коллекцию Chroma
print("Записей в базе:", vectorstore._collection.count())

# Проверка поиска знакомым запросом: метод вернёт пары (чанк, расстояние) —
# чем меньше dist, тем ближе
for doc, dist in vectorstore.similarity_search_with_score(
        "что означает код ошибки E-52", k=3):
    print(f"dist={dist:.3f} | стр. {doc.metadata['page']} | "
          f"{doc.page_content[:60]!r}")

Страниц: 3
Метаданные: velta_guide.pdf | стр. 0
Чанков: 9
Записей в базе: 9
dist=0.143 | стр. 1 | 'засорять чат при массовом сбое, и повторяет сообщение, если '
dist=0.207 | стр. 1 | 'при повторе обратитесь в сервис. Использовать адаптер сторон'
dist=0.231 | стр. 0 | 'Вельта. Руководство администратора\nРедакция 4.2. Сетевое обо'


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# Переменная vectorstore — из задания 3

# Пропуск 1: ретривер поверх векторной базы, k=3
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Локальный сервер llama.cpp — параметры знакомы по прошлым урокам
llm = ChatOpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
    model="qwen3.5",
    temperature=0.0,
    max_tokens=200,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

# Тот же системный промпт с заземлением, что в прошлом уроке
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
     "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
     "нет». Отвечай кратко, одним-двумя предложениями."),
    ("human", "Фрагменты базы знаний:\n{context}\n\nВопрос: {question}"),
])


def format_docs(docs):
    """Склеивает найденные Document в нумерованный контекст для промпта."""
    # Пропуск 2: верните строку вида "[1] текст\n[2] текст\n..."
    return "\n".join(f"[{i}] {doc.page_content}" for i, doc in enumerate(docs, 1))


# Пропуск 3: соберите цепочку — развилка, промпт, модель, разбор ответа
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

QUESTIONS = [
    "Что означает код ошибки E-52 и что с ним делать?",
    "Сколько лет действует гарантия на устройства и что она не покрывает?",
    "На каком языке запросов работает ревизия панели R3?",
    "Как часто интеграция Гросбух выгружает счета в бухгалтерию?",
    "Можно ли крепить точку доступа Пульсар-7 на улице зимой?",
]

for q in QUESTIONS:
    answer = rag_chain.invoke(q)
    # Цепочка возвращает только текст ответа — страницы достаём отдельным
    # вызовом ретривера; +1: в метаданных нумерация страниц с нуля
    pages = sorted({d.metadata["page"] + 1 for d in retriever.invoke(q)})
    print(f"\nВ: {q}\n   страницы: {pages}\n   О: {answer}")


В: Что означает код ошибки E-52 и что с ним делать?
   страницы: [2, 3]
   О: Код E-52 сигнализирует о чрезмерном нагреве процессора. Необходимо обеспечить продув устройства и убрать его от источников тепла.

В: Сколько лет действует гарантия на устройства и что она не покрывает?
   страницы: [2, 3]
   О: Гарантия на устройства действует три года со дня приобретения. Она не покрывает случаи залития, ударов, а также самостоятельное вскрытие устройства или использование адаптеров стороннего производителя.

В: На каком языке запросов работает ревизия панели R3?
   страницы: [1, 2]
   О: Ревизия панели R3 работает на языке запросов GraphQL.

В: Как часто интеграция Гросбух выгружает счета в бухгалтерию?
   страницы: [2, 3]
   О: Интеграция Гросбух выгружает счета и акты в бухгалтерию раз в сутки ночью.

В: Можно ли крепить точку доступа Пульсар-7 на улице зимой?
   страницы: [1, 3]
   О: Да, точку доступа Пульсар-7 можно крепить на улице зимой, так как она сохраняет работоспособность при 